In [1]:
import copy
import os
import time
import warnings

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from xgboost import XGBClassifier


## Phase 1: Data Foundation & Metadata Preparation
Loads HAM10000 metadata, encodes labels using LabelEncoder,
and formats 19-dimensional clinical metadata array.
- Age: median imputation + StandardScaler
- Sex, Localization: One-Hot Encoded
- Label mapping: akiec=0, bcc=1, bkl=2, df=3, mel=4, nv=5, vasc=6

In [2]:
csv_path = '../data/raw/HAM10000_metadata.csv'
folder_1 = '../data/raw/images/'
folder_2 = '../data/raw/images/'

# Load the CSV
df = pd.read_csv(csv_path)

# 2. Encode Labels 
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['dx'])

# Print Class Mapping for reference
class_mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
print(f"Class Mapping: {class_mapping}")

# 3. Secure Pathing 
image_paths = []
valid_indices = []

for idx, row in df.iterrows():
    file_name = f"{row['image_id']}.jpg"
    path_1 = os.path.join(folder_1, file_name)
    path_2 = os.path.join(folder_2, file_name)
    
    if os.path.exists(path_1):
        image_paths.append(path_1)
        valid_indices.append(idx)
    elif os.path.exists(path_2):
        image_paths.append(path_2)
        valid_indices.append(idx)

df_valid = df.iloc[valid_indices].copy()
df_valid['image_path'] = image_paths

# 4. Prepare Clinical Metadata
print("Formatting clinical metadata...")

# Handle missing ages with the median
df_valid['age'] = df_valid['age'].fillna(df_valid['age'].median())

# One-Hot Encode categorical text data
metadata_encoded = pd.get_dummies(df_valid[['sex', 'localization']])

# Scale the continuous age variable
scaler_age = StandardScaler()
age_scaled = scaler_age.fit_transform(df_valid[['age']])

# Insert the scaled age back into the metadata array
metadata_encoded.insert(0, 'age_scaled', age_scaled)

# Convert to final NumPy arrays
metadata_features = metadata_encoded.to_numpy(dtype=np.float64)
labels = df_valid['label'].to_numpy()

print("-" * 60)
print(f"Successfully secured {len(df_valid)} real image paths.")
print(f"Metadata perfectly formatted. Shape: {metadata_features.shape}")
print(f"Labels Shape: {labels.shape}")
print("-" * 60)

Class Mapping: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
Formatting clinical metadata...
------------------------------------------------------------
Successfully secured 10015 real image paths.
Metadata perfectly formatted. Shape: (10015, 19)
Labels Shape: (10015,)
------------------------------------------------------------


## Phase 2: Preprocessing & Dynamic Augmentation
- Hair removal (DullRazor algorithm via OpenCV morphological operations)
- Resize all images to 224×224
- 80/20 stratified train/test split (random_state=42)
- Training: aggressive augmentation (flip, rotation, color jitter)
- Testing: normalization only (no augmentation)
- Result: 8012 train / 2003 test images

In [3]:
# 1. Hair Removal Function (DullRazor)
def remove_hair(image):
    grayScale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    clean_image = cv2.inpaint(image, mask, 1, cv2.INPAINT_TELEA)
    return clean_image

# 2. Bulk Preprocess Images into RAM
print("Engaging CPU for Clinical Preprocessing (Hair Removal)...")
start_time = time.time()

processed_images = []
image_paths_list = df_valid['image_path'].tolist()

for i, path in enumerate(image_paths_list):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = remove_hair(img)             
    img = cv2.resize(img, (224, 224))  # Resize for ResNet50
    processed_images.append(img)
    
    if (i + 1) % 2000 == 0:
        print(f"Processed {i + 1}/{len(image_paths_list)} images...")

print(f"Preprocessing Complete! Time: {(time.time() - start_time) / 60:.2f} minutes.")
processed_images = np.array(processed_images)

# 3. Strict Train/Test Split
print("\nSplitting data into Train and Test sets...")
indices = np.arange(len(labels))

train_idx, test_idx = train_test_split(
    indices, test_size=0.20, stratify=labels, random_state=42
)

X_train_img, X_test_img = processed_images[train_idx], processed_images[test_idx]
y_train, y_test = labels[train_idx], labels[test_idx]
meta_train, meta_test = metadata_features[train_idx], metadata_features[test_idx]

print(f"Training Images: {len(X_train_img)} | Testing Images: {len(X_test_img)}")

# 4. Define the PyTorch Dataset
class SkinLesionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# 5. DEFINE TRANSFORMS 
# Training Pipeline: Aggressive randomization to build a robust feature extractor
train_transforms = transforms.Compose([
    transforms.ToPILImage(), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Testing Pipeline: Strict mathematical normalization only
test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 6. Instantiate Datasets and DataLoaders
train_dataset = SkinLesionDataset(X_train_img, y_train, transform=train_transforms)
test_dataset = SkinLesionDataset(X_test_img, y_test, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("\nPyTorch DataLoaders successfully mapped with Dynamic Augmentation!")

Engaging CPU for Clinical Preprocessing (Hair Removal)...
Processed 2000/10015 images...
Processed 4000/10015 images...
Processed 6000/10015 images...
Processed 8000/10015 images...
Processed 10000/10015 images...
Preprocessing Complete! Time: 8.83 minutes.

Splitting data into Train and Test sets...
Training Images: 8012 | Testing Images: 2003

PyTorch DataLoaders successfully mapped with Dynamic Augmentation!


## Phase 3: Unweighted Deep Fine-Tuning (ResNet-50)
- Entire network frozen initially
- layer3 and layer4 unfrozen for deep texture learning
- Discriminative learning rates: layer3=1e-5, layer4=1e-4, head=1e-3
- Standard CrossEntropyLoss WITHOUT class weights
  (balanced weights caused feature collapse to 11 dimensions)
- ReduceLROnPlateau scheduler
- Best validation accuracy: 85.87% (Epoch 10)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

print("\nLoading pre-trained ResNet50 & Resetting Architecture...")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze the entire network
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer3 and layer4 for deep texture learning
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace the final classification head
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)
model = model.to(device)

# THE FIX: Standard CrossEntropyLoss WITHOUT the 'balanced' weights!
criterion = nn.CrossEntropyLoss()

# Discriminative learning rates
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
])

# 2. The Training Engine
num_epochs = 10
print(f"\nStarting unweighted deep fine-tuning for {num_epochs} epochs...\n")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    start_time = time.time()
    
    # --- 1. TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = (correct_train / total_train) * 100

    # --- 2. VALIDATION PHASE ---
    model.eval()
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_acc = (correct_val / total_val) * 100
    epoch_time = (time.time() - start_time) / 60
    
    # Update the scheduler
    scheduler.step(epoch_val_acc)
    current_lr = optimizer.param_groups[1]['lr']
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {epoch_time:.2f} min | LR: {current_lr:.6f}")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | Val Acc: {epoch_val_acc:.2f}%")
    
    # Checkpoint
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        print(">>> New Best Validation Accuracy! Saving uncollapsed weights...")
        
    print("-" * 60)

# Restore the best model
model.load_state_dict(best_model_wts)
print(f"\nTraining Complete! Best Validation Accuracy: {best_val_acc:.2f}%")
print("ResNet50 is now deeply optimized WITHOUT feature collapse. Ready for extraction.")

Active Device: cpu

Loading pre-trained ResNet50 & Resetting Architecture...

Starting unweighted deep fine-tuning for 10 epochs...

Epoch [1/10] | Time: 34.96 min | LR: 0.000100
Train Loss: 0.7376 | Train Acc: 73.33% | Val Acc: 78.88%
>>> New Best Validation Accuracy! Saving uncollapsed weights...
------------------------------------------------------------
Epoch [2/10] | Time: 34.76 min | LR: 0.000100
Train Loss: 0.5723 | Train Acc: 78.99% | Val Acc: 82.53%
>>> New Best Validation Accuracy! Saving uncollapsed weights...
------------------------------------------------------------
Epoch [3/10] | Time: 39.85 min | LR: 0.000100
Train Loss: 0.5289 | Train Acc: 80.17% | Val Acc: 81.93%
------------------------------------------------------------
Epoch [4/10] | Time: 32.41 min | LR: 0.000100
Train Loss: 0.4664 | Train Acc: 82.20% | Val Acc: 82.43%
------------------------------------------------------------
Epoch [5/10] | Time: 30.98 min | LR: 0.000050
Train Loss: 0.4253 | Train Acc: 83.91

## Phase 4: Feature Extraction & Multimodal Fusion
- Classification head removed, 2048-dim features extracted
- Clean test_transforms used (no augmentation during extraction)
- 19-dim clinical metadata fused with 2048-dim visual features
- Final feature vector: 2067-dim per sample
- Saved: X_train_fused_optimized.npy (8012, 2067)
         X_test_fused_optimized.npy  (2003, 2067)

In [5]:
# 1. Strip the Classification Head
print("Removing the classification head to expose the deeply optimized 2048 feature layer...")
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval() 

# 2. Create Clean DataLoaders for Extraction
# CRITICAL: We apply the clean 'test_transforms' so the SVM receives undistorted features.
extract_train_dataset = SkinLesionDataset(X_train_img, y_train, transform=test_transforms)
extract_train_loader = DataLoader(extract_train_dataset, batch_size=32, shuffle=False)
extract_test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Extraction Function
def get_features(loader):
    features = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            outputs = feature_extractor(images)
            outputs = outputs.view(outputs.size(0), -1) 
            features.append(outputs.cpu().numpy())
    return np.vstack(features)

# 4. Execute Extraction
print("Extracting custom dermatological features from Training Set...")
X_train_features = get_features(extract_train_loader)

print("Extracting custom dermatological features from Testing Set...")
X_test_features = get_features(extract_test_loader)

print(f"\nExtracted Train Features: {X_train_features.shape}")
print(f"Extracted Test Features: {X_test_features.shape}")

# 5. Multimodal Fusion
print("\nFusing newly learned visual features with clinical metadata...")
X_train_fused = np.hstack((X_train_features, meta_train))
X_test_fused = np.hstack((X_test_features, meta_test))

print(f"Final Multimodal Train Shape: {X_train_fused.shape}")
print(f"Final Multimodal Test Shape: {X_test_fused.shape}")

# 6. Save Data to Hard Drive 
print("\nSaving final arrays to disk...")
np.save('../data/processed/X_train_fused_optimized.npy', X_train_fused)
np.save('../data/processed/X_test_fused_optimized.npy', X_test_fused)
np.save('../data/processed/y_train_optimized.npy', y_train)
np.save('../data/processed/y_test_optimized.npy', y_test)
print("Backup complete!")

Removing the classification head to expose the deeply optimized 2048 feature layer...
Extracting custom dermatological features from Training Set...
Extracting custom dermatological features from Testing Set...

Extracted Train Features: (8012, 2048)
Extracted Test Features: (2003, 2048)

Fusing newly learned visual features with clinical metadata...
Final Multimodal Train Shape: (8012, 2067)
Final Multimodal Test Shape: (2003, 2067)

Saving final arrays to disk...
Backup complete!


## Phase 5: SMOTE Balancing & High-Dimensional Classification
- Feature sanitization (NaN/Inf removal, constant feature removal)
- SMOTE applied to 2067-dim feature space (8012 → 37548 samples)
- StandardScaler + clip to [-10, 10] to prevent overflow
- SVM (RBF, C=15): F1-Macro 0.79
- MLP sklearn (512→128): F1-Macro 0.78
- Note: final models use svm_final.pkl (probability=True)

In [6]:
# 1. Load and Sanitize
print("Sanitizing features for mathematical stability...")
X_train_clean = np.nan_to_num(X_train_fused.astype(np.float64), nan=0.0, posinf=1e6, neginf=-1e6)
X_test_clean = np.nan_to_num(X_test_fused.astype(np.float64), nan=0.0, posinf=1e6, neginf=-1e6)

# Remove dead features to prevent divide-by-zero errors
selector = np.var(X_train_clean, axis=0) > 1e-10
X_train_clean = X_train_clean[:, selector]
X_test_clean = X_test_clean[:, selector]

# 2. APPLY SMOTE TO THE MATHEMATICAL FEATURES
print("\nApplying SMOTE to the 2067-dimensional feature space...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clean, y_train)
print(f"Training features expanded from {X_train_clean.shape[0]} to {X_train_resampled.shape[0]} samples.")

# 3. SCALE AND CLIP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled) # Fit strictly on the SMOTE data
X_test_scaled = scaler.transform(X_test_clean)

# Clip extreme values to prevent neural network overflows
X_train_scaled = np.clip(X_train_scaled, -10, 10)
X_test_scaled = np.clip(X_test_scaled, -10, 10)

class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# 4. TRAIN AND EVALUATE SVM
print("\nTraining the SMOTE-Enhanced SVM...")
svm_clf = SVC(kernel='rbf', C=15.0, gamma='scale', random_state=42)
svm_clf.fit(X_train_scaled, y_train_resampled)

print("\n" + "="*40)
print("        SVM CLASSIFICATION REPORT")
print("="*40)
print(classification_report(y_test, svm_clf.predict(X_test_scaled), target_names=class_names))

# 5. TRAIN AND EVALUATE NEURAL NETWORK (MLP)
print("\nTraining the SMOTE-Enhanced Neural Network (MLP)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    nn_clf = MLPClassifier(
        hidden_layer_sizes=(512, 128), 
        activation='relu', 
        solver='adam', 
        learning_rate_init=0.0005, 
        alpha=0.01, 
        max_iter=500, 
        early_stopping=True,
        random_state=42
    )
    nn_clf.fit(X_train_scaled, y_train_resampled)

print("\n" + "="*40)
print("   NEURAL NETWORK CLASSIFICATION REPORT")
print("="*40)
print(classification_report(y_test, nn_clf.predict(X_test_scaled), target_names=class_names))

Sanitizing features for mathematical stability...

Applying SMOTE to the 2067-dimensional feature space...
Training features expanded from 8012 to 37548 samples.

Training the SMOTE-Enhanced SVM...

        SVM CLASSIFICATION REPORT
              precision    recall  f1-score   support

       akiec       0.63      0.71      0.67        65
         bcc       0.78      0.75      0.76       103
         bkl       0.72      0.70      0.71       220
          df       1.00      0.78      0.88        23
         mel       0.65      0.63      0.64       223
          nv       0.93      0.95      0.94      1341
        vasc       1.00      0.86      0.92        28

    accuracy                           0.86      2003
   macro avg       0.82      0.77      0.79      2003
weighted avg       0.86      0.86      0.86      2003


Training the SMOTE-Enhanced Neural Network (MLP)...

   NEURAL NETWORK CLASSIFICATION REPORT
              precision    recall  f1-score   support

       akiec       0.

In [7]:
import pickle
import os

os.makedirs('../saved_models', exist_ok=True)

# sklearn MLP kaydet (baseline comparison)
with open('../saved_models/mlp_model_fused.pkl', 'wb') as f:
    pickle.dump(nn_clf, f)
print("sklearn MLP kaydedildi: mlp_model_fused.pkl")

# NOTE: Final SVM is saved separately as svm_final.pkl
# (probability=True, class_weight=balanced, trained on original data)
print("Final SVM: svm_final.pkl (probability=True, class_weight=balanced)")
print("Final MLP PyTorch: mlp_pytorch_final.pkl (Scenario 3, dropout=0.2)")

SVM kaydedildi.
MLP kaydedildi.
